## Environment Setup & BigQuery Configuration
We load the GOOGLE_PROJECT_ID from your .env file to establish a secure connection using the BigQuery Python SDK.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
import os
from dotenv import load_dotenv
from google.cloud import bigquery

warnings.filterwarnings('ignore')

# Set style for visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Load environment variables
load_dotenv()
PROJECT_ID = os.getenv('GOOGLE_PROJECT_ID')

# Initialize BigQuery client (using OAuth)
client = bigquery.Client(project=PROJECT_ID)
DATASET = "olist"

print(f"Connected to project: {PROJECT_ID}, dataset: {DATASET}")


## Cost Checker Setup

Run this as dry run

In [ ]:
from google.cloud import bigquery
import pandas_gbq

client = bigquery.Client()

def check_cost(query):
    job_config = bigquery.QueryJobConfig(dry_run=True)
    query_job = client.query(query, job_config=job_config)
    
    gb = query_job.total_bytes_processed / (1024**3)
    print(f"⚠️ This query will scan {gb:.4f} GB.")
    
    if gb > 0.03: # Set your limit to 0.03 GB (30MB)
        print("❌ TOO BIG! Optimize your SQL before running the next cell.")
        return False
    return True

## "Price Check" (Zero Cost)

Before you load your data, run this cell. It will tell you how much of your 1 TB quota you are about to use.

In [ ]:
# List your queries here
query_to_check = """
SELECT * FROM `olist.fct_sales`
WHERE time_id >= '2016-01-01'
"""

# CHECK THE PRICE
gb_to_spend = check_cost(query_to_check)

if gb_to_spend > 0.1: # 100MB Safety Limit
    print("⚠️ WARNING: This query is a bit large. Consider removing SELECT *")


## Cell 3: The "Payment" (Costs Money / Hits 1 TB Limit)
Only run this cell if the cell above gave you a "Price" you like.

In [ ]:
# This is the 'LIVE' execution. It scans data and 'bills' your quota.
df_sales = pandas_gbq.read_gbq(query_to_check, project_id=project_id)
print(f"✅ Data Loaded: {len(df_orders)} rows.")

## Cache to parquet
Run this tomorrow morning if you close your terminal to avoid paying to reload from bigQuery

In [ ]:
# Save the 'df_sales' DataFrame to a file on your machine
import os
print(os.getcwd())
df_sales.to_parquet("../data/my_sales_data.parquet")
print("✅ Data saved to my_sales_data.parquet")


In [ ]:
import pandas as pd

# Read the file back into a DataFrame
df_sales = pd.read_parquet("../data/my_sales_data.parquet")

print(f"✅ Data loaded from file: {len(df_sales)} rows.")
# Now you can use df_sales exactly as before

## Workzone from Pandas

You can run the following cell 1,000 times and it costs $0. It is only using your computer's RAM now.

In [ ]:
# Everything here is FREE because the data is already in Python
total_revenue = df_sales['total_payment_value'].sum()
print(f"Total Revenue: R$ {total_revenue:,.2f}")

# Plotting is also FREE
df_sales['total_payment_value'].hist()